# Agent: Humans Producer (Phase 1, No MQTT)

This notebook implements only Phase 1:
- local movement in a 1000x1000 meter area
- boundary reflection with overshoot mirroring
- illegal event generation (`event_type="illegal_act"`)
- max 1 event per human per step
- 1 Hz simulation cadence

No MQTT publish/subscribe is included in this phase.

# Agent: Humans Producer (Phase 1, No MQTT)

This notebook implements only Phase 1:
- local movement in a 1000x1000 meter area
- boundary reflection with overshoot mirroring
- illegal event generation (`event_type=\"illegal_act\"`)
- max 1 event per human per step
- 1 Hz simulation cadence

No MQTT publish/subscribe is included in this phase.

In [ ]:
# Setup / imports
from __future__ import annotations

# Standard-library imports used for simulation math/timing/randomization
import colorsys
import math
import random
import time
from dataclasses import dataclass
from typing import Any

# Notebook helper to render widgets/maps inside the output area
from IPython.display import display

# anymap-ts map widget for live map visualization
from anymap_ts import Map
# Project config loader (reads config.yaml and env vars)
from simulated_city.config import load_config

In [ ]:
# Config load + Phase 1 constants

# Load shared project configuration from config.yaml/.env
cfg = load_config()

# Parken Stadion (Copenhagen) center used as visual anchor for the map
PARKEN_LAT = 55.7029
PARKEN_LON = 12.5720

# Local simulation area boundaries in meters (0..1000 on both axes)
AREA_MIN = 0.0
AREA_MAX = 1000.0

# Movement/loop settings for the phase-1 agent
STEP_LENGTH_M = 7.0          # Each move is exactly 7 meters
STEP_DELAY_S = 1.0           # 1 timestep per real-world second
HUMAN_COUNT = 10             # Number of humans in this simulation
EVENT_PROBABILITY = 0.25     # 25% illegal event chance per human per step
TOTAL_STEPS = 60             # Total simulation steps for this notebook run
EVENT_TYPE = "illegal_act"   # Event type is fixed by design

# Quick startup printout for confidence/debugging
print(f"Config loaded. Primary MQTT host: {cfg.mqtt.host}")
print(f"Phase 1 run: humans={HUMAN_COUNT}, steps={TOTAL_STEPS}, delay={STEP_DELAY_S}s")

Config loaded. Primary MQTT host: 127.0.0.1
Phase 1 run: humans=10, steps=60, delay=1.0s


In [ ]:
# Helper functions: local meter coordinates, boundary reflection, and colors

def local_xy_to_lnglat(x_m: float, y_m: float, center_lat: float, center_lon: float) -> tuple[float, float]:
    """Approximate local meters (x east, y north) to (lng, lat)."""
    # Approximate meters-per-degree values; sufficient for small local offsets.
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lon = 111_320.0 * math.cos(math.radians(center_lat))

    # Convert local meter offsets to geographic deltas.
    lat = center_lat + (y_m / meters_per_deg_lat)
    lon = center_lon + (x_m / meters_per_deg_lon)
    return lon, lat


def reflect_axis(position: float, delta: float, low: float = AREA_MIN, high: float = AREA_MAX) -> tuple[float, float]:
    """Reflect one axis and mirror overshoot into [low, high]."""
    # First apply the proposed movement on this axis.
    new_position = position + delta
    new_delta = delta

    # If we left bounds, mirror overshoot back in and flip direction.
    # Loop handles rare large overshoot values robustly.
    while new_position < low or new_position > high:
        if new_position > high:
            overshoot = new_position - high
            new_position = high - overshoot
            new_delta = -new_delta
        elif new_position < low:
            overshoot = low - new_position
            new_position = low + overshoot
            new_delta = -new_delta

    return new_position, new_delta


def generate_unique_colors(count: int) -> list[str]:
    """Create random-but-unique colors for human markers."""
    # Start with evenly spaced hues so colors are distinct.
    hues = [index / count for index in range(count)]
    # Shuffle to avoid predictable color ordering between humans.
    random.shuffle(hues)

    colors: list[str] = []
    for hue in hues:
        # Convert HSV to RGB for vivid, readable marker colors.
        r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
        colors.append(f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}")

    return colors

In [ ]:
# Human model + initialization

# Dataclass keeps each simulated human's state in one simple object.
@dataclass
class Human:
    human_id: str
    name: str
    x: float
    y: float
    marker_color: str
    criminal_count: int = 0


# Fixed list of 10 names (design requires unique full-name IDs).
NAMES = [
    "Ava Jensen",
    "Liam Nielsen",
    "Emma Larsen",
    "Noah Pedersen",
    "Sofia Madsen",
    "Oliver Christensen",
    "Clara Andersen",
    "William Thomsen",
    "Freja Rasmussen",
    "Lucas Mortensen",
]

# Build one unique marker color per human.
colors = generate_unique_colors(HUMAN_COUNT)
humans: list[Human] = []
for index in range(HUMAN_COUNT):
    humans.append(
        Human(
            human_id=NAMES[index],
            name=NAMES[index],
            # Start each human at a random position inside [0, 1000] x [0, 1000].
            x=random.uniform(AREA_MIN, AREA_MAX),
            y=random.uniform(AREA_MIN, AREA_MAX),
            marker_color=colors[index],
        )
    )

# Print initialized state so you can inspect the random start distribution.
print("Initialized humans:")
for human in humans:
    print(f"- {human.human_id}: x={human.x:.1f}, y={human.y:.1f}, color={human.marker_color}")

Initialized humans:
- Ava Jensen: x=747.7, y=135.4, color=#cd3cf2
- Liam Nielsen: x=716.1, y=853.4, color=#f2a93c
- Emma Larsen: x=569.1, y=289.7, color=#3c85f2
- Noah Pedersen: x=97.8, y=836.9, color=#f23c3c
- Sofia Madsen: x=857.8, y=523.5, color=#3cf2f2
- Oliver Christensen: x=685.4, y=807.6, color=#3cf285
- Clara Andersen: x=652.7, y=142.6, color=#603cf2
- William Thomsen: x=542.4, y=241.6, color=#60f23c
- Freja Rasmussen: x=114.3, y=894.9, color=#cdf23c
- Lucas Mortensen: x=713.7, y=548.8, color=#f23ca9


In [ ]:
# Map setup (anymap)

# Create map centered on Parken for visual context.
m = Map(center=(PARKEN_LON, PARKEN_LAT), zoom=15.8, height="650px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")

# Render the initial human positions as map markers.
for human in humans:
    # Convert local simulation coordinates to approximate lng/lat around map center.
    lng, lat = local_xy_to_lnglat(human.x - 500.0, human.y - 500.0, PARKEN_LAT, PARKEN_LON)
    # Use human_id as stable marker name so updates can target the same person.
    m.add_marker(lng, lat, name=human.human_id, color=human.marker_color, popup=human.name)

# Show map widget in notebook output.
display(m)
print("Map displayed. Run the simulation cell to start.")

Map displayed. Run the simulation cell to start.


In [ ]:
# 1 Hz simulation loop: movement + event generation (no MQTT)

# Dictionary keyed by step -> list of events generated in that step.
events_by_step: dict[int, list[dict[str, Any]]] = {}

for step in range(TOTAL_STEPS):
    # Timestamp start of loop iteration to maintain near-1Hz pacing.
    step_started = time.perf_counter()
    step_events: list[dict[str, Any]] = []

    for human in humans:
        # Continuous movement with random direction each step.
        angle = random.uniform(0.0, 2.0 * math.pi)
        dx = STEP_LENGTH_M * math.cos(angle)
        dy = STEP_LENGTH_M * math.sin(angle)

        # Reflect independently per axis with mirrored overshoot.
        human.x, _ = reflect_axis(human.x, dx)
        human.y, _ = reflect_axis(human.y, dy)

        # Update marker position to match new simulated coordinates.
        lng, lat = local_xy_to_lnglat(human.x - 500.0, human.y - 500.0, PARKEN_LAT, PARKEN_LON)
        m.remove_marker(human.human_id)
        m.add_marker(lng, lat, name=human.human_id, color=human.marker_color, popup=human.name)

        # Illegal event generation (max 1 event per human per step).
        if random.random() < EVENT_PROBABILITY:
            step_events.append(
                {
                    "step": step,
                    # Contracted event_id format from design spec.
                    "event_id": f"{step}:{human.human_id}",
                    "human_id": human.human_id,
                    "name": human.name,
                    "x": human.x,
                    "y": human.y,
                    "event_type": EVENT_TYPE,
                }
            )

    # Save all events generated in the current timestep.
    events_by_step[step] = step_events
    print(f"step={step:03d} events={len(step_events)}")

    # Sleep only the remaining time to keep total loop time near 1 second.
    elapsed = time.perf_counter() - step_started
    time.sleep(max(0.0, STEP_DELAY_S - elapsed))

print("Phase 1 simulation complete.")

step=000 events=1
step=001 events=4
step=002 events=4
step=003 events=3
step=004 events=4
step=005 events=2
step=006 events=2
step=007 events=3
step=008 events=2
step=009 events=4
step=010 events=3
step=011 events=4
step=012 events=3
step=013 events=7
step=014 events=2
step=015 events=3
step=016 events=2
step=017 events=5
step=018 events=2
step=019 events=4
step=020 events=2
step=021 events=2
step=022 events=2
step=023 events=2
step=024 events=2
step=025 events=2
step=026 events=1
step=027 events=3
step=028 events=3
step=029 events=3
step=030 events=1
step=031 events=2
step=032 events=2
step=033 events=4
step=034 events=2
step=035 events=3
step=036 events=2
step=037 events=1
step=038 events=0
step=039 events=4
step=040 events=1
step=041 events=2
step=042 events=3
step=043 events=2
step=044 events=5


KeyboardInterrupt: 

In [ ]:
# Quick verification checks for Phase 1

# Flatten event list across all steps to evaluate uniqueness contract.
all_event_ids = [event["event_id"] for events in events_by_step.values() for event in events]
unique_event_ids = set(all_event_ids)

# Compare counts: if equal, there are no duplicate event IDs.
print(f"Total events: {len(all_event_ids)}")
print(f"Unique event_ids: {len(unique_event_ids)}")
print(f"event_id uniqueness ok: {len(all_event_ids) == len(unique_event_ids)}")

# Print a small sample for manual inspection.
if all_event_ids:
    print("Sample event_ids:")
    for event_id in list(unique_event_ids)[:10]:
        print(" -", event_id)